# 01 · Data understanding

**Questions this notebook answers**

1. What is in the dataset, and where does it come from?
2. Is it structurally clean (schema, missing values, duplicates, value ranges)?
3. Are there signs about how the data was generated that limit what we can claim?
4. How rare is fraud, and how is the data split for honest evaluation?
5. What business problem are we solving, and how do we price a mistake?

All logic lives in `src/fraud_e2e/`; this notebook calls it and explains the results.
Every number in the text below is computed in this notebook, not typed by hand.

> **Kaggle edition.** Generated from the project repository by `scripts/build_kaggle.py`. Attach the dataset [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) (CC0) before running. The first code cell installs the project's helper package into `/kaggle/working`.

In [ ]:
# Bootstrap: write the project package into the working directory (auto-generated).
import os, sys
from pathlib import Path
WORK = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
FILES = {'src/fraud_e2e/__init__.py': 'def hello() -> str:\n    return "Hello from fraud-e2e!"\n', 'src/fraud_e2e/config.py': '"""Project-wide constants: paths, seed, column groups.\n\nPaths resolve to the repo layout locally and to the Kaggle input mount when the\ncode runs inside a Kaggle notebook, so the same notebooks work in both places.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nSEED = 42\n\n# Repo root = two levels above this file (src/fraud_e2e/config.py).\n_REPO_ROOT = Path(__file__).resolve().parents[2]\n# Kaggle mounts datasets under /kaggle/input/<slug>/ (layout varies), outputs go to\n# /kaggle/working. Env overrides exist only so the Kaggle build can be tested locally.\n_KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))\n_KAGGLE_WORKING = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))\n_RAW_NAME = "credit_card_fraud_2026.csv"\nON_KAGGLE = _KAGGLE_INPUT.is_dir() and "FRAUD_E2E_ROOT" not in os.environ\n\nif ON_KAGGLE:\n    ROOT = _KAGGLE_WORKING\n    _found = sorted(_KAGGLE_INPUT.rglob(_RAW_NAME))\n    if not _found:\n        raise FileNotFoundError(f"Attach the dataset: no {_RAW_NAME} under {_KAGGLE_INPUT}")\n    RAW_CSV = _found[0]\nelse:\n    ROOT = Path(os.environ.get("FRAUD_E2E_ROOT", _REPO_ROOT))\n    RAW_CSV = ROOT / "data" / "raw" / _RAW_NAME\n\nSPLITS_CSV = ROOT / "data" / "splits" / "splits.csv"\nREPORTS_DIR = ROOT / "reports"\nFIGURES_DIR = REPORTS_DIR / "figures"\nCONFIGS_DIR = ROOT / "configs"\n\n# SHA-256 of the canonical CSV (see data/raw/SOURCE.md). Loading fails loudly if\n# the file on disk is not byte-identical to the one the analysis was built on.\nRAW_SHA256 = "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61"\n\nID_COL = "transaction_id"\nTARGET = "is_fraud"\nAMOUNT_COL = "amount_usd"\n\nCATEGORICAL_COLS = [\n    "merchant_category",\n    "card_type",\n    "auth_method",\n    "channel",\n    "device_type",\n]\n\nBOOLEAN_COLS = [\n    "is_foreign_transaction",\n    "is_new_merchant",\n    "used_vpn",\n    "ip_country_mismatch",\n    "billing_shipping_mismatch",\n    "is_ai_generated_scam_attempt",\n]\n\n# Integer-valued columns with few distinct values (counts / clock fields).\nDISCRETE_COLS = [\n    "txn_count_last_24h",\n    "cvv_retry_count",\n    "prior_disputes",\n    "time_of_day_hour",\n    "day_of_week",\n]\n\nCONTINUOUS_COLS = [\n    "amount_usd",\n    "hours_since_last_txn",\n    "distance_from_home_km",\n    "card_age_months",\n    "customer_age",\n    "account_balance_usd",\n    "velocity_score",\n    "merchant_risk_score",\n]\n\nNUMERIC_COLS = CONTINUOUS_COLS + DISCRETE_COLS\nFEATURE_COLS = CATEGORICAL_COLS + BOOLEAN_COLS + NUMERIC_COLS\n', 'src/fraud_e2e/cost.py': '"""Business cost model for an alert/decline decision.\n\nEach transaction scored by the model ends in one of four outcomes:\n\n  TP  fraud, flagged      -> analyst review cost\n  FP  legit, flagged      -> analyst review cost + good-customer friction cost\n  FN  fraud, not flagged  -> fraction of the amount lost + fixed chargeback handling\n  TN  legit, not flagged  -> 0\n\nAll parameters live in configs/cost.yaml and are passed in; nothing is hard-coded\nhere. Values in that file are *assumptions* that must be confirmed (see\nnotes/01_cost_assumptions.md) before they drive any decision.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport yaml\n\nfrom fraud_e2e import config as C\n\n\n@dataclass(frozen=True)\nclass CostParams:\n    review_cost_per_alert: float  # USD per flagged transaction (TP and FP)\n    fp_friction_cost: float  # USD extra per wrongly flagged good customer\n    fn_loss_fraction: float  # share of amount_usd lost when fraud is missed (0..1)\n    fn_fixed_cost: float  # USD per missed fraud (chargeback handling)\n    status: str = "proposed"  # "proposed" until Nat confirms, then "confirmed"\n\n    def __post_init__(self):\n        for name in ("review_cost_per_alert", "fp_friction_cost", "fn_fixed_cost"):\n            if getattr(self, name) < 0:\n                raise ValueError(f"{name} must be >= 0")\n        if not 0 <= self.fn_loss_fraction <= 1:\n            raise ValueError("fn_loss_fraction must be within [0, 1]")\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef load_cost_params(path: Path | None = None) -> CostParams:\n    path = Path(path or C.CONFIGS_DIR / "cost.yaml")\n    with open(path) as f:\n        raw = yaml.safe_load(f)\n    return CostParams(**raw["params"], status=raw.get("status", "proposed"))\n\n\ndef outcome_costs(y_true, flagged, amount, p: CostParams) -> np.ndarray:\n    """Per-transaction cost in USD for a given set of flag decisions."""\n    y = np.asarray(y_true).astype(bool)\n    f = np.asarray(flagged).astype(bool)\n    a = np.asarray(amount, dtype=float)\n    cost = np.zeros(len(y))\n    cost[f] += p.review_cost_per_alert\n    cost[f & ~y] += p.fp_friction_cost\n    miss = y & ~f\n    cost[miss] += p.fn_loss_fraction * a[miss] + p.fn_fixed_cost\n    return cost\n\n\ndef total_cost(y_true, flagged, amount, p: CostParams) -> float:\n    return float(outcome_costs(y_true, flagged, amount, p).sum())\n\n\ndef reference_costs(y_true, amount, p: CostParams) -> dict:\n    """Costs of the two trivial policies every model must beat."""\n    y = np.asarray(y_true)\n    return {\n        "flag_nothing": total_cost(y, np.zeros_like(y, bool), amount, p),\n        "flag_everything": total_cost(y, np.ones_like(y, bool), amount, p),\n    }\n', 'src/fraud_e2e/data.py': '"""Loading the raw CSV and data-quality checks.\n\nEvery check returns a plain dict so results can be dumped to JSON and quoted in\ndocuments without retyping numbers by hand.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n# Allowed domain per column. Numeric ranges are *plausibility* bounds chosen\n# before looking at the data (e.g. an hour must be 0-23); they are not fitted.\nEXPECTED_LEVELS: dict[str, set[str]] = {\n    "merchant_category": {\n        "Online Retail",\n        "Groceries",\n        "Restaurants",\n        "Electronics",\n        "Fuel",\n        "Travel",\n        "Utilities",\n        "Healthcare",\n        "Gaming",\n        "Streaming",\n        "Gift Cards",\n        "Crypto Exchange",\n    },\n    "card_type": {"Visa", "Mastercard", "Amex", "RuPay", "Discover"},\n    "auth_method": {"3D Secure", "OTP", "PIN", "Biometric", "No Authentication"},\n    "channel": {"Online", "POS", "Contactless", "In-App", "ATM"},\n    "device_type": {\n        "Android Phone",\n        "iPhone",\n        "POS Terminal",\n        "Windows PC",\n        "Mac",\n        "Tablet",\n        "Smart Watch",\n        "ATM Machine",\n    },\n}\n\nRANGES: dict[str, tuple[float, float]] = {\n    "amount_usd": (0.01, 1e6),\n    "hours_since_last_txn": (0, 24 * 365),\n    "txn_count_last_24h": (0, 500),\n    "distance_from_home_km": (0, 20_100),  # half Earth\'s circumference\n    "card_age_months": (0, 600),\n    "customer_age": (18, 110),\n    "account_balance_usd": (0, 1e8),\n    "cvv_retry_count": (0, 10),\n    "velocity_score": (0, 100),\n    "time_of_day_hour": (0, 23),\n    "day_of_week": (0, 6),\n    "merchant_risk_score": (0, 100),\n    "prior_disputes": (0, 100),\n}\n\n# Cross-column combinations that would be odd for real card traffic. They are\n# reported, not removed: the dataset is used as published.\nPLAUSIBILITY_RULES: dict[str, str] = {\n    "atm_channel_on_non_atm_device": "channel == \'ATM\' and device_type != \'ATM Machine\'",\n    "atm_device_on_non_atm_channel": "device_type == \'ATM Machine\' and channel != \'ATM\'",\n    "card_present_channel_on_desktop": (\n        "channel in [\'POS\', \'Contactless\'] and device_type in [\'Windows PC\', \'Mac\']"\n    ),\n    "shipping_mismatch_at_atm": "channel == \'ATM\' and billing_shipping_mismatch",\n    "amount_exceeds_account_balance": "amount_usd > account_balance_usd",\n}\n\n\ndef sha256_of(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_raw(path: Path | None = None, verify_hash: bool = True) -> pd.DataFrame:\n    """Read the canonical CSV with explicit dtypes."""\n    path = Path(path or C.RAW_CSV)\n    if verify_hash:\n        got = sha256_of(path)\n        if got != C.RAW_SHA256:\n            raise ValueError(f"{path} sha256 {got} != expected {C.RAW_SHA256}")\n    dtypes = {c: "category" for c in C.CATEGORICAL_COLS}\n    dtypes.update({c: "bool" for c in C.BOOLEAN_COLS})\n    df = pd.read_csv(path, dtype=dtypes)\n    df[C.TARGET] = df[C.TARGET].astype("int8")\n    return df\n\n\ndef _check(name: str, passed: bool, **detail) -> dict:\n    return {"check": name, "passed": bool(passed), **detail}\n\n\ndef quality_checks(df: pd.DataFrame) -> list[dict]:\n    """Structural checks that do not look at the label\'s relationship with features."""\n    out: list[dict] = []\n    expected_cols = [C.ID_COL, *C.FEATURE_COLS, C.TARGET]\n    out.append(\n        _check(\n            "columns_match_schema",\n            set(df.columns) == set(expected_cols),\n            missing=sorted(set(expected_cols) - set(df.columns)),\n            unexpected=sorted(set(df.columns) - set(expected_cols)),\n        )\n    )\n    out.append(_check("row_count", True, n_rows=len(df), n_cols=df.shape[1]))\n\n    n_null = df.isna().sum()\n    out.append(\n        _check(\n            "no_missing_values",\n            int(n_null.sum()) == 0,\n            missing_by_col={k: int(v) for k, v in n_null.items() if v},\n        )\n    )\n\n    ids = df[C.ID_COL]\n    out.append(_check("id_unique", ids.is_unique, n_unique=int(ids.nunique())))\n    out.append(\n        _check(\n            "id_contiguous_1_to_n",\n            bool((ids.sort_values().to_numpy() == range(1, len(df) + 1)).all()),\n        )\n    )\n\n    n_dup = int(df.drop(columns=C.ID_COL).duplicated().sum())\n    out.append(_check("no_duplicate_rows_excluding_id", n_dup == 0, n_duplicates=n_dup))\n    n_dup_x = int(df[C.FEATURE_COLS].duplicated().sum())\n    out.append(_check("no_duplicate_feature_vectors", n_dup_x == 0, n_duplicates=n_dup_x))\n\n    labels = sorted(df[C.TARGET].unique().tolist())\n    out.append(\n        _check(\n            "target_is_binary",\n            labels == [0, 1],\n            values=labels,\n            n_positive=int(df[C.TARGET].sum()),\n            positive_rate=float(df[C.TARGET].mean()),\n        )\n    )\n\n    for col, allowed in EXPECTED_LEVELS.items():\n        seen = set(df[col].astype(str).unique())\n        out.append(\n            _check(\n                f"levels_{col}",\n                seen <= allowed,\n                unexpected=sorted(seen - allowed),\n                n_levels=len(seen),\n            )\n        )\n\n    for col, (lo, hi) in RANGES.items():\n        s = df[col]\n        bad = int(((s < lo) | (s > hi)).sum())\n        out.append(\n            _check(\n                f"range_{col}",\n                bad == 0,\n                n_out_of_range=bad,\n                observed_min=float(s.min()),\n                observed_max=float(s.max()),\n                allowed=[lo, hi],\n            )\n        )\n    return out\n\n\ndef plausibility_checks(df: pd.DataFrame) -> list[dict]:\n    """Count rows matching each implausible cross-column combination."""\n    out = []\n    for name, expr in PLAUSIBILITY_RULES.items():\n        n = int(df.eval(expr).sum())\n        out.append({"rule": name, "expression": expr, "n_rows": n, "share_of_rows": n / len(df)})\n    return out\n', 'src/fraud_e2e/plots.py': '"""Matplotlib helpers with one consistent, readable style.\n\nColour roles (fixed, never cycled): legit = blue, fraud = orange, reference\nlines = neutral grey. Every plot sets axis labels with units.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\nLEGIT = "#2a78d6"\nFRAUD = "#eb6834"\nNEUTRAL = "#8a8985"\nINK = "#0b0b0b"\nINK_2 = "#52514e"\nGRID = "#e4e3df"\n\n\ndef set_style() -> None:\n    plt.rcParams.update(\n        {\n            "figure.dpi": 110,\n            "savefig.dpi": 150,\n            "savefig.bbox": "tight",\n            "figure.facecolor": "#fcfcfb",\n            "axes.facecolor": "#fcfcfb",\n            "axes.edgecolor": GRID,\n            "axes.labelcolor": INK_2,\n            "axes.titlecolor": INK,\n            "axes.titlesize": 12,\n            "axes.titleweight": "bold",\n            "axes.labelsize": 10,\n            "axes.spines.top": False,\n            "axes.spines.right": False,\n            "axes.grid": True,\n            "grid.color": GRID,\n            "grid.linewidth": 0.8,\n            "xtick.color": INK_2,\n            "ytick.color": INK_2,\n            "legend.frameon": False,\n            "font.size": 10,\n        }\n    )\n\n\ndef save(fig, name: str, folder: Path = C.FIGURES_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.png"\n    fig.savefig(path)\n    return path\n\n\ndef rate_bars(table: pd.DataFrame, base_rate: float, ax=None, title: str | None = None):\n    """Horizontal bars of fraud rate per level with 95% Wilson CI whiskers."""\n    t = table.sort_values("fraud_rate")\n    ax = ax or plt.subplots(figsize=(6, 0.35 * len(t) + 1.2))[1]\n    y = np.arange(len(t))\n    ax.barh(y, t["fraud_rate"] * 100, color=FRAUD, height=0.6, zorder=2)\n    err = [(t["fraud_rate"] - t["ci_low"]) * 100, (t["ci_high"] - t["fraud_rate"]) * 100]\n    ax.errorbar(\n        t["fraud_rate"] * 100,\n        y,\n        xerr=err,\n        fmt="none",\n        ecolor=INK_2,\n        elinewidth=1,\n        capsize=2,\n        zorder=3,\n    )\n    ax.axvline(base_rate * 100, color=NEUTRAL, ls="--", lw=1, zorder=1)\n    ax.set_yticks(y, [f"{lvl}  (n={n:,})" for lvl, n in zip(t["level"], t["n"])])\n    ax.set_xlabel("Fraud rate (%) — bars: 95% Wilson CI, dashed: overall rate")\n    ax.grid(axis="y", visible=False)\n    ax.set_title(title or t["feature"].iloc[0])\n    return ax\n\n\ndef binned_rate_line(\n    table: pd.DataFrame, base_rate: float, xlabel: str, ax=None, logx: bool = False\n):\n    """Fraud rate across quantile bins of a numeric feature, CI as a band."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    x = table["bin_mid"].astype(float)\n    ax.fill_between(x, table["ci_low"] * 100, table["ci_high"] * 100, color=FRAUD, alpha=0.18, lw=0)\n    ax.plot(x, table["fraud_rate"] * 100, color=FRAUD, lw=2, marker="o", ms=4)\n    ax.axhline(base_rate * 100, color=NEUTRAL, ls="--", lw=1)\n    if logx:\n        ax.set_xscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("Fraud rate (%)")\n    ax.set_ylim(bottom=0)\n    return ax\n\n\ndef class_hist(\n    df: pd.DataFrame,\n    col: str,\n    xlabel: str,\n    ax=None,\n    bins: int = 40,\n    logx: bool = False,\n    target: str = C.TARGET,\n):\n    """Overlaid histograms for legit vs fraud, each class normalised to 100%."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    x = df[col].astype(float)\n    if logx:\n        edges = np.geomspace(max(x.min(), 1e-2), x.max(), bins + 1)\n    else:\n        edges = np.histogram_bin_edges(x, bins=bins)\n    for label, colour, name in [(0, LEGIT, "Legit"), (1, FRAUD, "Fraud")]:\n        xs = x[df[target] == label]\n        ax.hist(\n            xs,\n            bins=edges,\n            weights=np.full(len(xs), 100 / len(xs)),\n            histtype="step",\n            lw=2,\n            color=colour,\n            label=f"{name} (n={len(xs):,})",\n        )\n    if logx:\n        ax.set_xscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("Share of class in bin (%)")\n    ax.legend()\n    return ax\n\n\ndef auc_forest(table: pd.DataFrame, ax=None):\n    """Univariate ROC AUC per feature with bootstrap CI; 0.5 = no signal."""\n    t = table.sort_values("separation")\n    ax = ax or plt.subplots(figsize=(6, 0.33 * len(t) + 1.2))[1]\n    y = np.arange(len(t))\n    colours = [FRAUD if a > 0.5 else LEGIT for a in t["auc"]]\n    ax.hlines(y, t["auc_ci_low"], t["auc_ci_high"], color=INK_2, lw=1.5)\n    ax.scatter(t["auc"], y, color=colours, s=40, zorder=3, edgecolor="#fcfcfb", lw=1.5)\n    ax.axvline(0.5, color=NEUTRAL, ls="--", lw=1)\n    ax.set_yticks(y, t["feature"])\n    ax.set_xlabel("Univariate ROC AUC (95% bootstrap CI); >0.5 orange = higher value → more fraud")\n    ax.grid(axis="y", visible=False)\n    return ax\n', 'src/fraud_e2e/reporting.py': '"""Write analysis results as committed artifacts (JSON / CSV).\n\nDocuments and the website quote numbers only from these files, so every\nartifact carries the data hash and seed it was produced with.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n\ndef _default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, np.ndarray):\n        return o.tolist()\n    if isinstance(o, pd.DataFrame):\n        return o.to_dict(orient="records")\n    raise TypeError(f"Not JSON serialisable: {type(o)}")\n\n\ndef write_json(payload: dict, name: str, folder: Path = C.REPORTS_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.json"\n    body = {"_meta": {"data_sha256": C.RAW_SHA256, "seed": C.SEED}, **payload}\n    path.write_text(json.dumps(body, indent=2, default=_default, sort_keys=False) + "\\n")\n    return path\n\n\ndef write_csv(df: pd.DataFrame, name: str, folder: Path = C.REPORTS_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.csv"\n    df.to_csv(path, index=False, float_format="%.6g")\n    return path\n\n\ndef read_json(name: str, folder: Path = C.REPORTS_DIR) -> dict:\n    return json.loads((folder / f"{name}.json").read_text())\n', 'src/fraud_e2e/splits.py': '"""Locked data splits.\n\nDesign (fixed before any label-based EDA, see notes/00_preregistration.md):\n  * 20% stratified holdout, touched once at the very end of modelling.\n  * The remaining 80% ("dev") gets repeated stratified 5-fold CV assignments\n    (3 repeats, different shuffles) so fold-to-fold variance can be reported.\nThe assignments are written to data/splits/splits.csv and committed; every\nlater stage reads that file instead of re-splitting.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nfrom fraud_e2e import config as C\n\nHOLDOUT_FRACTION = 0.20\nN_FOLDS = 5\nN_REPEATS = 3\n\n\ndef make_splits(df: pd.DataFrame, seed: int = C.SEED) -> pd.DataFrame:\n    """Return one row per transaction: split (\'dev\'/\'holdout\') and fold_r{k} for dev rows."""\n    ids = df[C.ID_COL].to_numpy()\n    y = df[C.TARGET].to_numpy()\n    dev_ids, _ = train_test_split(ids, test_size=HOLDOUT_FRACTION, stratify=y, random_state=seed)\n    out = pd.DataFrame({C.ID_COL: ids, "split": "holdout"})\n    out.loc[out[C.ID_COL].isin(dev_ids), "split"] = "dev"\n\n    dev_mask = out["split"].eq("dev").to_numpy()\n    y_dev = y[dev_mask]\n    for r in range(N_REPEATS):\n        fold = np.full(len(out), -1, dtype=int)\n        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed + r)\n        dev_fold = np.empty(dev_mask.sum(), dtype=int)\n        for k, (_, val_idx) in enumerate(skf.split(np.zeros(len(y_dev)), y_dev)):\n            dev_fold[val_idx] = k\n        fold[dev_mask] = dev_fold\n        out[f"fold_r{r}"] = fold\n    return out\n\n\ndef save_splits(splits: pd.DataFrame, path: Path = C.SPLITS_CSV) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    splits.to_csv(path, index=False)\n\n\ndef load_splits(path: Path = C.SPLITS_CSV) -> pd.DataFrame:\n    return pd.read_csv(path)\n\n\ndef attach_splits(df: pd.DataFrame, splits: pd.DataFrame | None = None) -> pd.DataFrame:\n    splits = load_splits() if splits is None else splits\n    merged = df.merge(splits, on=C.ID_COL, how="left", validate="one_to_one")\n    if merged["split"].isna().any():\n        raise ValueError("Some transactions have no split assignment")\n    return merged\n\n\ndef dev_only(df: pd.DataFrame) -> pd.DataFrame:\n    """The only slice that label-based EDA and model selection may look at."""\n    if "split" not in df.columns:\n        df = attach_splits(df)\n    return df[df["split"].eq("dev")].reset_index(drop=True)\n', 'src/fraud_e2e/stats.py': '"""Small statistics helpers for a rare-event target.\n\nWith only a few hundred positives, every rate and score here is reported with\nan interval. Intervals are Wilson (for proportions) or stratified bootstrap\npercentile intervals (for AUC), both seeded for reproducibility.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats as ss\n\nfrom fraud_e2e import config as C\n\n\ndef wilson_ci(k: np.ndarray | int, n: np.ndarray | int, alpha: float = 0.05):\n    """Wilson score interval for k successes out of n. Vectorised."""\n    k = np.asarray(k, dtype=float)\n    n = np.asarray(n, dtype=float)\n    z = ss.norm.ppf(1 - alpha / 2)\n    p = np.divide(k, n, out=np.zeros_like(k), where=n > 0)\n    denom = 1 + z**2 / n\n    centre = (p + z**2 / (2 * n)) / denom\n    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom\n    lo = np.where(k == 0, 0.0, np.clip(centre - half, 0, 1))  # exact bounds at the edges\n    hi = np.where(k == n, 1.0, np.clip(centre + half, 0, 1))\n    return lo, hi\n\n\ndef rate_table(df: pd.DataFrame, col: str, target: str = C.TARGET) -> pd.DataFrame:\n    """Fraud rate per level of `col` with 95% Wilson CI and lift vs the overall rate."""\n    g = df.groupby(col, observed=True)[target].agg(n="size", n_fraud="sum").reset_index()\n    g["fraud_rate"] = g["n_fraud"] / g["n"]\n    g["ci_low"], g["ci_high"] = wilson_ci(g["n_fraud"].to_numpy(), g["n"].to_numpy())\n    base = df[target].mean()\n    g["lift"] = g["fraud_rate"] / base\n    g.insert(0, "feature", col)\n    g = g.rename(columns={col: "level"})\n    g["level"] = g["level"].astype(str)\n    return g.sort_values("fraud_rate", ascending=False).reset_index(drop=True)\n\n\ndef chi2_association(df: pd.DataFrame, col: str, target: str = C.TARGET) -> dict:\n    """Chi-square test of independence + Cramér\'s V between a discrete column and the target."""\n    table = pd.crosstab(df[col], df[target])\n    chi2, p, dof, _ = ss.chi2_contingency(table, correction=False)\n    n = table.to_numpy().sum()\n    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))\n    return {\n        "feature": col,\n        "chi2": float(chi2),\n        "dof": int(dof),\n        "p_value": float(p),\n        "cramers_v": float(v),\n        "n_levels": int(table.shape[0]),\n    }\n\n\ndef rank_auc(x: np.ndarray, y: np.ndarray) -> float:\n    """ROC AUC of score x for binary labels y via the Mann-Whitney U statistic (ties averaged)."""\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y).astype(bool)\n    n_pos, n_neg = y.sum(), (~y).sum()\n    ranks = ss.rankdata(x)\n    return float((ranks[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))\n\n\ndef bootstrap_auc_ci(\n    x: np.ndarray, y: np.ndarray, n_boot: int = 1000, seed: int = C.SEED, alpha: float = 0.05\n) -> tuple[float, float]:\n    """Stratified bootstrap percentile CI for rank_auc.\n\n    Positives and negatives are resampled separately so every replicate keeps the class counts.\n    """\n    rng = np.random.default_rng(seed)\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y).astype(bool)\n    pos, neg = x[y], x[~y]\n    lab = np.r_[np.ones(len(pos), bool), np.zeros(len(neg), bool)]\n    aucs = np.empty(n_boot)\n    for b in range(n_boot):\n        xb = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]\n        aucs[b] = rank_auc(xb, lab)\n    return float(np.quantile(aucs, alpha / 2)), float(np.quantile(aucs, 1 - alpha / 2))\n\n\ndef univariate_numeric(\n    df: pd.DataFrame,\n    cols: list[str],\n    target: str = C.TARGET,\n    n_boot: int = 1000,\n    seed: int = C.SEED,\n) -> pd.DataFrame:\n    """Per numeric column: class medians/IQR and univariate ROC AUC with bootstrap CI.\n\n    AUC > 0.5 means larger values are associated with fraud; AUC < 0.5 the opposite.\n    `separation` = |AUC - 0.5| ranks features regardless of direction.\n    """\n    y = df[target].to_numpy()\n    rows = []\n    for i, c in enumerate(cols):\n        x = df[c].to_numpy(dtype=float)\n        auc = rank_auc(x, y)\n        lo, hi = bootstrap_auc_ci(x, y, n_boot=n_boot, seed=seed + i)\n        q = lambda a, p: float(np.quantile(a, p))  # noqa: E731\n        xp, xn = x[y == 1], x[y == 0]\n        rows.append(\n            {\n                "feature": c,\n                "median_fraud": q(xp, 0.5),\n                "iqr_fraud": [q(xp, 0.25), q(xp, 0.75)],\n                "median_legit": q(xn, 0.5),\n                "iqr_legit": [q(xn, 0.25), q(xn, 0.75)],\n                "auc": auc,\n                "auc_ci_low": lo,\n                "auc_ci_high": hi,\n                "separation": abs(auc - 0.5),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("separation", ascending=False).reset_index(drop=True)\n\n\ndef binned_rate(\n    df: pd.DataFrame, col: str, n_bins: int = 10, target: str = C.TARGET\n) -> pd.DataFrame:\n    """Fraud rate by quantile bin of a numeric column, with Wilson CI.\n\n    Low-cardinality integer columns (<= 15 distinct values) are grouped by exact value\n    instead, with the top 1% folded into the 99th-percentile value so no bin is tiny.\n    """\n    if df[col].nunique() <= 15:\n        cap = df[col].quantile(0.99)\n        x = df[col].clip(upper=cap)\n        g = df.groupby(x)[target].agg(n="size", n_fraud="sum").reset_index()\n        g["bin_mid"] = g["bin_left"] = g["bin_right"] = g[col].astype(float)\n    else:\n        bins = pd.qcut(df[col], q=n_bins, duplicates="drop")\n        g = df.groupby(bins, observed=True)[target].agg(n="size", n_fraud="sum").reset_index()\n        g["bin_mid"] = [iv.mid for iv in g[col]]\n        g["bin_left"] = [iv.left for iv in g[col]]\n        g["bin_right"] = [iv.right for iv in g[col]]\n    g["fraud_rate"] = g["n_fraud"] / g["n"]\n    g["ci_low"], g["ci_high"] = wilson_ci(g["n_fraud"].to_numpy(), g["n"].to_numpy())\n    g.insert(0, "feature", col)\n    return g.drop(columns=col)\n', 'configs/cost.yaml': '# Cost assumptions for threshold selection (P2). See notes/01_cost_assumptions.md.\n# status must be "confirmed" before these values drive any decision.\nstatus: proposed\nparams:\n  review_cost_per_alert: 5.0   # USD per flagged transaction (analyst review / step-up auth)\n  fp_friction_cost: 10.0       # USD extra per good customer wrongly flagged (friction, support, churn risk)\n  fn_loss_fraction: 1.0        # share of amount_usd lost on a missed fraud (1.0 = issuer absorbs full amount)\n  fn_fixed_cost: 15.0          # USD per missed fraud (chargeback handling)\n# Ranges for the P2 sensitivity analysis (min, max)\nsensitivity:\n  review_cost_per_alert: [1.0, 20.0]\n  fp_friction_cost: [0.0, 50.0]\n  fn_loss_fraction: [0.5, 1.0]\n  fn_fixed_cost: [0.0, 30.0]\n'}
for rel, text in FILES.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
sys.path.insert(0, str(WORK / "src"))
from fraud_e2e import config as C
from fraud_e2e.data import load_raw
from fraud_e2e.splits import make_splits, save_splits
if not C.SPLITS_CSV.exists():
    save_splits(make_splits(load_raw()))
print("fraud_e2e ready; data:", C.RAW_CSV)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency

from fraud_e2e import config as C
from fraud_e2e import plots as P
from fraud_e2e import stats as S
from fraud_e2e.cost import load_cost_params
from fraud_e2e.data import load_raw, plausibility_checks, quality_checks
from fraud_e2e.reporting import write_json
from fraud_e2e.splits import attach_splits

P.set_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 1. Source and provenance

- **Dataset:** *Credit Card Fraud Detection 2026* by Udit Jain, Kaggle, licence **CC0 (public domain)**.
  <https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026>
- **Provenance:** the data card does not say whether the rows are real bank transactions or synthetic.
  This project treats it as a **public CC0 dataset of unspecified provenance** and makes no claim
  that results transfer to real card traffic.
- The loader checks the file's SHA-256 so every result is tied to one exact file.

In [ ]:
df = load_raw()
print(f"Loaded {C.RAW_CSV.name}: {df.shape[0]:,} rows x {df.shape[1]} columns (sha256 verified)")
df.head()

## 2. Columns

One row is one card transaction. `transaction_id` is a row identifier (never used as a feature)
and `is_fraud` is the target. The 24 features fall into four groups.

In [ ]:
group = {
    **{c: "categorical" for c in C.CATEGORICAL_COLS},
    **{c: "boolean" for c in C.BOOLEAN_COLS},
    **{c: "discrete (count/clock)" for c in C.DISCRETE_COLS},
    **{c: "continuous" for c in C.CONTINUOUS_COLS},
    C.ID_COL: "identifier",
    C.TARGET: "target",
}
schema = pd.DataFrame(
    {
        "column": df.columns,
        "group": [group[c] for c in df.columns],
        "dtype": [str(t) for t in df.dtypes],
        "n_unique": [df[c].nunique() for c in df.columns],
        "example": [df[c].iloc[0] for c in df.columns],
    }
)
schema

In [ ]:
df[C.CONTINUOUS_COLS + C.DISCRETE_COLS].describe().T.round(2)

## 3. Structural quality checks

Schema, missing values, duplicates, allowed category levels and plausible numeric ranges.
The ranges were written down as common-sense bounds (e.g. hour 0–23, age ≥ 18), not fitted to the data.
These checks do not look at the label's relationship with features, so they use all rows.

In [ ]:
qc = quality_checks(df)
qc_table = pd.DataFrame([{"check": q["check"], "passed": q["passed"]} for q in qc])
n_failed = int((~qc_table["passed"]).sum())
print(f"{len(qc_table)} checks, {n_failed} failed")
qc_table[~qc_table["passed"]] if n_failed else qc_table.head(12)

## 4. Plausibility of cross-column combinations

Some combinations would be unusual for real card traffic, for example a cash-machine
transaction made from a smart watch, or an in-store (POS/contactless) payment made from a desktop PC.
We count them; we do **not** remove them, because the dataset is used as published.

In [ ]:
plaus = pd.DataFrame(plausibility_checks(df))
plaus["share_%"] = (plaus["share_of_rows"] * 100).round(2)
plaus[["rule", "n_rows", "share_%"]]

In [ ]:
ct = pd.crosstab(df["channel"], df["device_type"], normalize="index") * 100
fig, ax = P.plt.subplots(figsize=(8, 3.2))
im = ax.imshow(ct.to_numpy(), cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks(range(ct.shape[1]), ct.columns, rotation=30, ha="right")
ax.set_yticks(range(ct.shape[0]), ct.index)
ax.set_xlabel("Device type")
ax.set_ylabel("Channel")
ax.grid(False)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(
            j,
            i,
            f"{ct.iat[i, j]:.0f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if ct.iat[i, j] > 25 else P.INK,
        )
fig.colorbar(im, ax=ax, label="% of channel's transactions")
ax.set_title("Device mix is nearly the same for every channel")
P.save(fig, "p1_channel_vs_device")
P.plt.show()

In [ ]:
# How similar is the device mix across channels? Max absolute difference (percentage points)
# between any channel's device share and the overall device share.
overall_dev = df["device_type"].value_counts(normalize=True) * 100
max_pp_gap = float((ct - overall_dev[ct.columns]).abs().to_numpy().max())
_, p_chan_dev, _, _ = chi2_contingency(pd.crosstab(df["channel"], df["device_type"]))
display(
    Markdown(
        f"Largest gap between a channel's device share and the overall device share: "
        f"**{max_pp_gap:.1f} percentage points**; chi-square test of channel × device independence "
        f"p = **{p_chan_dev:.2g}**."
    )
)

**Reading.** Channel and device type look close to statistically independent: an ATM
transaction is about as likely to come from a phone as an online one is. Real card data would show
strong structure here (ATM transactions on ATM machines). This pattern is *consistent with* the
columns having been sampled independently, which is typical of synthetic data. We do not conclude
that the data is synthetic, but it reinforces the rule of not claiming real-world performance.

## 5. Class balance and locked splits

The split was created **before** any label-based analysis by `scripts/make_splits.py` and saved to
`data/splits/splits.csv` (see `notes/00_preregistration.md`):

- **holdout** (20%, stratified) — scored once at the end of modelling;
- **dev** (80%) — 5-fold stratified CV, repeated 3 times with different shuffles.

In [ ]:
dfs = attach_splits(df)
rows = []
for name, part in [("all", dfs), *dfs.groupby("split")]:
    k, n = int(part[C.TARGET].sum()), len(part)
    lo, hi = S.wilson_ci(k, n)
    rows.append(
        {
            "subset": name,
            "n": n,
            "n_fraud": k,
            "fraud_rate_%": 100 * k / n,
            "ci95_low_%": 100 * float(lo),
            "ci95_high_%": 100 * float(hi),
        }
    )
balance = pd.DataFrame(rows)
balance.round(3)

In [ ]:
fold_rows = []
dev = dfs[dfs["split"] == "dev"]
for r in range(3):
    for k, part in dev.groupby(f"fold_r{r}"):
        fold_rows.append(
            {"repeat": r, "fold": k, "n": len(part), "n_fraud": int(part[C.TARGET].sum())}
        )
folds = pd.DataFrame(fold_rows)
folds.pivot(index="fold", columns="repeat", values="n_fraud").add_prefix("fraud in repeat ")

In [ ]:
fig, ax = P.plt.subplots(figsize=(5.5, 2.4))
counts = df[C.TARGET].value_counts().sort_index()
ax.barh(["Legit", "Fraud"], counts.to_numpy(), color=[P.LEGIT, P.FRAUD], height=0.55)
for i, v in enumerate(counts.to_numpy()):
    ax.text(v, i, f"  {v:,} ({100 * v / counts.sum():.2f}%)", va="center", color=P.INK)
ax.set_xlabel("Number of transactions")
ax.set_xlim(0, counts.max() * 1.25)
ax.grid(axis="y", visible=False)
ax.set_title(f"Fraud is rare: about 1 in {round(counts.sum() / counts[1])} transactions")
P.save(fig, "p1_class_balance")
P.plt.show()

## 6. Business framing

**Who uses the model?** A card issuer's fraud team. Each incoming transaction gets a fraud score;
transactions above a threshold are flagged for review or step-up authentication.

**Questions the project answers**

1. How well can fraud be ranked from these features (PR-AUC, with uncertainty)?
2. At which threshold is the total cost of mistakes lowest, and how sensitive is that choice
   to the cost assumptions?
3. Which features drive the score, overall and for a single transaction?
4. How would we notice if incoming data drifted away from the training data?

**Pricing a mistake.** Two errors cost money in different ways:

| Outcome | What happens | Cost |
|---|---|---|
| True positive | fraud caught | review cost |
| **False positive** | good customer bothered (declined / asked to verify) | review cost + friction cost |
| **False negative** | fraud missed | share of the transaction amount lost + chargeback handling |
| True negative | nothing | 0 |

Because a missed fraud costs (part of) its amount, an expensive fraud matters more than a cheap one.
The parameters live in `configs/cost.yaml`. They are **assumptions proposed for review**; they will
not drive any decision until their status is `confirmed`.

In [ ]:
cost = load_cost_params()
pd.Series(cost.to_dict(), name="value").to_frame()

## 7. Limitations recorded before modelling

- **No customer or card ID.** Customer-level features (spend history, typical merchants) cannot be
  built, and the split cannot be grouped by customer. If one customer appears in both train and
  validation, CV is optimistic by an unknown amount.
- **No real timestamp.** Only hour-of-day and day-of-week exist, so there is no out-of-time
  validation. Drift in the monitoring phase will be *simulated*.
- **Unspecified provenance.** Results describe this dataset, not real-world card fraud.
- **Small positive class.** With a few hundred fraud rows, every metric is reported with an interval.

In [ ]:
summary = {
    "n_rows": len(df),
    "n_columns": df.shape[1],
    "n_features": len(C.FEATURE_COLS),
    "feature_groups": {
        "categorical": C.CATEGORICAL_COLS,
        "boolean": C.BOOLEAN_COLS,
        "discrete": C.DISCRETE_COLS,
        "continuous": C.CONTINUOUS_COLS,
    },
    "quality_checks": qc,
    "n_quality_checks": len(qc),
    "n_quality_checks_failed": n_failed,
    "plausibility_checks": plausibility_checks(df),
    "channel_device_independence": {"chi2_p_value": p_chan_dev, "max_share_gap_pp": max_pp_gap},
    "class_balance": balance.to_dict(orient="records"),
    "fold_fraud_counts": folds.to_dict(orient="records"),
    "cost_params": cost.to_dict(),
}
path = write_json(summary, "p1_data_understanding")
print(f"wrote {path.relative_to(C.ROOT)}")

## Summary

In [ ]:
b_all = balance.set_index("subset").loc["all"]
b_dev = balance.set_index("subset").loc["dev"]
b_hold = balance.set_index("subset").loc["holdout"]
worst = plaus.sort_values("n_rows", ascending=False).iloc[0]
display(
    Markdown(f"""
- **Size:** {len(df):,} transactions, {len(C.FEATURE_COLS)} features
  ({len(C.CATEGORICAL_COLS)} categorical, {len(C.BOOLEAN_COLS)} boolean,
  {len(C.NUMERIC_COLS)} numeric), plus an ID and the target.
- **Quality:** {len(qc) - n_failed} of {len(qc)} structural checks pass — no missing values,
  no duplicate rows, all categories and ranges valid.
- **Plausibility:** combinations unusual for real card traffic are common (largest:
  `{worst["rule"]}`, {int(worst["n_rows"]):,} rows = {100 * worst["share_of_rows"]:.1f}%), and channel
  and device look nearly independent (max gap {max_pp_gap:.1f} pp). Consistent with synthetic
  generation; provenance stays "unspecified".
- **Fraud rate:** {int(b_all["n_fraud"])} of {int(b_all["n"]):,} = {b_all["fraud_rate_%"]:.3f}%
  (95% CI {b_all["ci95_low_%"]:.2f}–{b_all["ci95_high_%"]:.2f}%).
- **Locked split:** dev {int(b_dev["n"]):,} rows / {int(b_dev["n_fraud"])} fraud;
  holdout {int(b_hold["n"]):,} rows / {int(b_hold["n_fraud"])} fraud; each CV validation fold holds
  {folds["n_fraud"].min()}–{folds["n_fraud"].max()} fraud.
- **Next:** notebook 02 explores which features carry signal, on the dev split only.
""")
)